In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 관련 ################################################
# 결측치 처리
from sklearn.impute import SimpleImputer
# 표준화
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
# 인코더
from sklearn.preprocessing import LabelEncoder
# 원핫 인코더
from sklearn.preprocessing import OneHotEncoder

# 학습 모델 성능 관련 #######################################
# 원하는 비율로 데이터를 나누기 위해
from sklearn.model_selection import train_test_split
# K-Fold 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
# 학습 곡선
from sklearn.model_selection import learning_curve
# 하이퍼 파라미터 튜닝
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 모델 성능 평가 ########################
# 회귀용
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score
# 분류용
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.metrics import roc_auc_score
from sklearn.metrics import ConfusionMatrixDisplay

# 피처 선택 #########################
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import RFE
from sklearn.inspection import permutation_importance

# 학습 모델 ########################
# 분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier 
from sklearn.naive_bayes import GaussianNB
from catboost import CatBoostClassifier

# 회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import BayesianRidge
from catboost import CatBoostRegressor

# 결정트리를 시각화할 수 있는 라이브러리
from sklearn.tree import plot_tree

# 차원 축소
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import TSNE

# 연관 규칙 학습
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import association_rules

# 군집
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.mixture import GaussianMixture
from sklearn.cluster import MeanShift, estimate_bandwidth

# 파이프라인
from sklearn.pipeline import Pipeline

# KDE를 그리기 위한 통계값을 구할 수 있는 함수
from scipy.stats import gaussian_kde

# 피어슨 상관 계수 (연속형 수치형 데이터 vs 연속형 수치형 데이터)
from scipy.stats import pearsonr
# 카이제곱 검증 (범주형 데이터 vs 범주형 데이터, 순위X)
from scipy.stats import chi2_contingency
# 스피어만 상관계수 (범주형 데이터 vs 범주형 데이터, 순위O)
from scipy.stats import spearmanr
# 포인트 이분 상관계수 (범주형 데이터 vs 연속형 수치형 데이터)
from scipy.stats import pointbiserialr

# 객체를 파일에 저장
import pickle

# 불필요한 경고 뜨지 않게
import warnings
warnings.filterwarnings('ignore')

### 데이터를 불러온다.

In [2]:
train_df = pd.read_csv('data/bike_sharing_train5.csv')
train_df

,temp,humidity,log_count,year,log_windspeed,season_1,season_2,season_3,season_4,weather_1,...,hour_14,hour_15,hour_16,hour_17,hour_18,hour_19,hour_20,hour_21,hour_22,hour_23
0,9.84,81,2.833213,1,0.000000,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,9.02,80,3.713572,1,0.000000,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,9.02,80,3.496508,1,0.000000,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,9.84,75,2.639057,1,0.000000,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,9.84,75,0.693147,1,0.000000,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10881,15.58,50,5.820083,0,3.295937,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
10882,14.76,57,5.488938,0,2.772670,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
10883,13.94,61,5.129899,0,2.772670,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
10884,13.94,61,4.867534,0,1.946367,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [3]:
X = train_df.drop('log_count', axis=1)
y = train_df['log_count']

In [4]:
display(X)
display(y)

,temp,humidity,year,log_windspeed,season_1,season_2,season_3,season_4,weather_1,weather_3,...,hour_14,hour_15,hour_16,hour_17,hour_18,hour_19,hour_20,hour_21,hour_22,hour_23
0,9.84,81,1,0.000000,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,9.02,80,1,0.000000,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,9.02,80,1,0.000000,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,9.84,75,1,0.000000,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,9.84,75,1,0.000000,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10881,15.58,50,0,3.295937,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
10882,14.76,57,0,2.772670,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
10883,13.94,61,0,2.772670,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
10884,13.94,61,0,1.946367,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


0        2.833213
1        3.713572
2        3.496508
3        2.639057
4        0.693147
           ...   
10881    5.820083
10882    5.488938
10883    5.129899
10884    4.867534
10885    4.488636
Name: log_count, Length: 10886, dtype: float64

In [5]:
scaler1 = StandardScaler()
X_scaled = scaler1.fit_transform(X)
X_scaled

array([[-1.33366069,  0.99321305,  1.00386564, ..., -0.20909337,
        -0.20909337, -0.20909337],
       [-1.43890721,  0.94124921,  1.00386564, ..., -0.20909337,
        -0.20909337, -0.20909337],
       [-1.43890721,  0.94124921,  1.00386564, ..., -0.20909337,
        -0.20909337, -0.20909337],
       ...,
       [-0.80742813, -0.04606385, -0.99614925, ...,  4.78255235,
        -0.20909337, -0.20909337],
       [-0.80742813, -0.04606385, -0.99614925, ..., -0.20909337,
         4.78255235, -0.20909337],
       [-0.91267464,  0.21375537, -0.99614925, ..., -0.20909337,
        -0.20909337,  4.78255235]], shape=(10886, 43))

### KNeighborsRegressor 하이퍼 파라미터 튜닝

In [6]:
def objective(trial) :
    params = {
        # 이웃의 개수
        # 회귀에서는 이웃들의 평균(또는 가중 평균)을 구한다.
        'n_neighbors' : trial.suggest_int('n_neighbors', 1, 100),
        # 가중치 결정 방식
        # uniform : 모두에게 동일한 가중치를 부여
        # distance : 거리가 가까운 데이터의 가중치를 더 부여
        'weights' : trial.suggest_categorical('weights', ['uniform', 'distance']),
        # 거리 측정방식
        'metric' : trial.suggest_categorical('metric', ['euclidean', 'manhattan', 'minkowski', 'chebyshev']),
        # 이웃 탐색 알고리즘
        'algorithm' : trial.suggest_categorical('algorithm', ['auto', 'ball_tree', 'kd_tree', 'brute']),
        # 트리의 최소 노드 크기
        'leaf_size' : trial.suggest_int('leaf_size', 10, 60),
        # 병렬 처리
        'n_jobs' : -1
    }

    # 거리측정방식이 minkowski 라면
    if params['metric'] == 'minkowski' :
        params['p'] = trial.suggest_float('p', 1.0, 5.0)

    # 모델 생성
    model = KNeighborsRegressor(**params)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    score = cross_val_score(model, X_scaled, y, cv=kf, scoring='r2').mean()
    return score

sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, pruner=MedianPruner(n_warmup_steps=10))
study.optimize(objective, n_trials=50, show_progress_bar=True)

knn_params = study.best_params.copy()
knn_score = study.best_value

print(f'최적 파라미터 : {knn_params}')
print(f'최고 R2 점수 : {knn_score}')

  0%|          | 0/50 [00:00<?, ?it/s]

최적 파라미터 : {'n_neighbors': 46, 'weights': 'distance', 'metric': 'manhattan', 'algorithm': 'ball_tree', 'leaf_size': 46}
최고 R2 점수 : 0.8110910210099137


### LinearRegression 하이퍼 파라미터 튜닝

In [7]:
def objective(trial) :
    params = {
        # 모형에 상수항을 계산할지 여부
        # 데이터가 이미 원점을 지나도록 전처리되었다면 False로 설정하기도 함.
        'fit_intercept' : trial.suggest_categorical('fit_intercept', [True, False]),
        # 학습 시 입력 데이터를 복하해서 사용할지 여부
        # False일 경우 원본 데이터가 변경될 수 있지만 메모리를 절약한다.
        'copy_X' : trial.suggest_categorical('copy_X', [True, False]),
        # 회귀 계수(기울기)를 무조건 양수로 제한할지 여부
        # 특정 도메인에서 음수 계수가 나오면 안 될 때 사용한다.
        'positive' : trial.suggest_categorical('positive', [True, False]),
        # 모든 CPU 코어를 사용하겠다
        'n_jobs' : -1
    }

    model = LinearRegression(**params)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    score = cross_val_score(model, X_scaled, y, cv=kf, scoring='r2').mean()
    return score

sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, pruner=MedianPruner(n_warmup_steps=10))
study.optimize(objective, n_trials=50, show_progress_bar=True)

lr_params = study.best_params.copy()
lr_score = study.best_value

print(f'최적 파라미터 : {lr_params}')
print(f'최고 R2 점수 : {lr_score}')

  0%|          | 0/50 [00:00<?, ?it/s]

최적 파라미터 : {'fit_intercept': True, 'copy_X': False, 'positive': False}
최고 R2 점수 : 0.8291325676996888


### Ridge 하이퍼 파라미터 튜닝

In [8]:
def objective(trial) :
    params = {
        # 규제의 강도를 결정하는 핵심 파라미터
        # 값이 클수록 규제가 강해져 계수가 0에 가까워진다(모델이 단순해진다)
        'alpha' : trial.suggest_float('alpha', 1e-3, 100.0, log=True),
        # 절편(혹은 바이어스)를 계산할지 여부
        'fit_intercept' : trial.suggest_categorical('fit_intercept', [True, False]),
        # 계산에 사용할 알고리즘
        # auto : 데이터에 따라 자동 선택
        # svd : 특이값 분해 (데이터가 클 때 유리)
        # cholesky : 표준적인 계산 방식
        # sag / saga : 확률적 평균 경사 하강법 (대용량 데이터에 최적)
        'solver' : trial.suggest_categorical('solver', ['auto', 'svd', 'cholesky', 'sag', 'saga']),
        # 반복 횟수 (solver가 sag, saga일 경우 사용)
        'max_iter' : 1000,
        # 랜덤 시드
        'random_state' : 42
    }

    model = Ridge(**params)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    score = cross_val_score(model, X_scaled, y, cv=kf, scoring='r2').mean()
    return score

sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, pruner=MedianPruner(n_warmup_steps=10))
study.optimize(objective, n_trials=50, show_progress_bar=True)

ridge_params = study.best_params.copy()
ridge_score = study.best_value

print(f'최적 파라미터 : {ridge_params}')
print(f'최적 R2 점수 : {ridge_score}')

  0%|          | 0/50 [00:00<?, ?it/s]

최적 파라미터 : {'alpha': 12.601010727086686, 'fit_intercept': True, 'solver': 'saga'}
최적 R2 점수 : 0.8293762246481837


### Lasso 하이퍼 파라미터 튜닝

In [9]:
def objective(trial) :
    params = {
        # L1 규제 강도
        # 0에 가까우면 일반 선형 회귀, 커질 수록 더 많은 변수의 계수가 0이 된다.
        'alpha' : trial.suggest_float('alpha', 1e-4, 10.0, log=True),
        # fit_intercept : 절편 계산 여부
        'fit_intercept' : trial.suggest_categorical('fit_intercept', [True, False]),
        # 계수를 업데이트하는 방식
        # cyclic : 변수 순서대로 업데이트
        # random : 무작위 업데이트 (빠르게 끝날 수도 있지만 최적의 가중치를 찾지 못할 수도 있다)
        'selection' : trial.suggest_categorical('selection', ['cyclic', 'random']),
        # 최대 반복 횟수
        # Lasso는 오랫동안 학습해야 하는 경우가 많기 때문에 많이 주세요
        'max_iter' : 2000,
        # 랜덤시드
        'random_state' : 42
    }

    model = Lasso(**params)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    score = cross_val_score(model, X_scaled, y, cv=kf, scoring='r2').mean()
    return score

sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, pruner=MedianPruner(n_warmup_steps=10))
study.optimize(objective, n_trials=50, show_progress_bar=True)

lasso_params = study.best_params.copy()
lasso_score = study.best_value

print(f'최적 파라미터 : {lasso_params}')
print(f'최고 R2 점수 : {lasso_score}')

  0%|          | 0/50 [00:00<?, ?it/s]

최적 파라미터 : {'alpha': 0.0001128445313593244, 'fit_intercept': True, 'selection': 'random'}
최고 R2 점수 : 0.8293752145637404


### ElasticNet 하이퍼 파라미터 튜닝

In [10]:
def objective(trial) :
    params = {
        # 규제 강도 : L1 + L2 합산 강도
        'alpha' : trial.suggest_float('alpha', 1e-4, 10.0, log=True),
        # 규제 비율
        # 0에 가까우면 Ridge와 비슷, 1에 가까우면 Lasso와 비슷
        # 0.5면 L1과 L2를 반반씩 섞은 형태이다
        'l1_ratio' : trial.suggest_float('l1_ratio', 0.0, 1.0),
        # 절편 계산 여부
        'fit_intercept' : trial.suggest_categorical('fit_intercept', [True, False]),
        # 반복횟수
        'max_iter' : 2000,
        # 랜덤 시드
        'random_state' : 42
    }

    model = ElasticNet(**params)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    score = cross_val_score(model, X_scaled, y, cv=kf, scoring='r2').mean()
    return score

sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, pruner=MedianPruner(n_warmup_steps=10))
study.optimize(objective, n_trials=50, show_progress_bar=True)

en_params = study.best_params.copy()
en_score = study.best_value

print(f'최적 파라미터 : {en_params}')
print(f'최고 R2 점수 : {en_score}')

  0%|          | 0/50 [00:00<?, ?it/s]

최적 파라미터 : {'alpha': 0.0009162724884836632, 'l1_ratio': 0.566218510535344, 'fit_intercept': True}
최고 R2 점수 : 0.8293775549520621


### SVR 하이퍼 파라미터 튜닝

In [11]:
def objective(trial) :
    params = {
        # 데이터의 비 선형성을 처리하는 방식
        'kernel' : trial.suggest_categorical('kernel', ['linear', 'poly', 'rbf', 'sigmoid']),
        # 규제 강도의 역수
        # 크면 오차를 적게 허용(과적합 위험), 작으면 부드러운 경계(과소적합 위험)
        'C' : trial.suggest_float('C', 1e-3, 100.0, log=True),
        # 튜브의 너비
        # 이 범위내의 오차는 손실로 계산하지 않는다.
        'epsilon' : trial.suggest_float('epsilon', 0.01, 1.0, log=True),
        # 커널의 영향력의 범위 (rbf, poly, sigmoid에서 중요)
        # 'scale'은 피처의 수에 반비례, auto의 피처 수의 역수
        'gamma' : trial.suggest_categorical('gamma', ['scale', 'auto']),
        # poly 커널 사용시 다항식의 차수
        'degree' : trial.suggest_int('degree', 2, 5) if trial.params.get('kernel') == 'poly' else 3,
        # 계산을 위한 메모리 할당 (MB 단위)
        'cache_size' : 1000
    }

    model = SVR(**params)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    score = cross_val_score(model, X_scaled, y, cv=kf, scoring='r2').mean()
    return score

sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, pruner=MedianPruner(n_warmup_steps=10))
study.optimize(objective, n_trials=50, show_progress_bar=True)

svr_params = study.best_params.copy()
svr_score = study.best_value

print(f'최적 파라미터 : {svr_params}')
print(f'최고 R2 점수 : {svr_score}')

  0%|          | 0/50 [00:00<?, ?it/s]

최적 파라미터 : {'kernel': 'linear', 'C': 2.820926771119946, 'epsilon': 0.5333740234646415, 'gamma': 'scale'}
최고 R2 점수 : 0.8262362258078356


### DesionTreeRegressor 하이퍼 파라미터 튜닝

In [12]:
def objective(trial) :
    params = {
        # 분할의 품질을 측정하는 함수
        # squared_error : 평균 제곱 오차
        # absolute_error : 평균 절대 오차
        # friedman_mse : mse를 조금 개선한 방식
        'criterion' : trial.suggest_categorical('criterion', ['squared_error', 'friedman_mse', 'absolute_error']),
        # 트리의 최대 깊이
        # 너무 깊으면 과적합, 너무 낮으면 과소적합 발생
        'max_depth' : trial.suggest_int('max_depth', 3, 20),
        # 노드를 분할하기 위해 필요한 최소 샘플 수
        'min_samples_split' : trial.suggest_int('min_samples_split', 2, 20),
        # 리프 노드(말단 노드)가 되기 위해 필요한 최소 샘플 수
        'min_samples_leaf' : trial.suggest_int('min_samples_leaf', 1, 20),
        # 최적의 분할을 찾기 위해 고려할 피처의 개수
        'max_features' : trial.suggest_categorical('max_features', [None, 'sqrt', 'log2']),
        # 랜덤 시드
        'random_state' : 42
    }

    model = DecisionTreeRegressor(**params)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    score = cross_val_score(model, X_scaled, y, cv=kf, scoring='r2').mean()
    return score

sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, pruner=MedianPruner(n_warmup_steps=10))
study.optimize(objective, n_trials=50, show_progress_bar=True)

dt_params = study.best_params.copy()
dt_score = study.best_value

print(f'최적 파라미터 : {dt_params}')
print(f'최고 R2 점수 : {dt_score}')

  0%|          | 0/50 [00:00<?, ?it/s]

최적 파라미터 : {'criterion': 'friedman_mse', 'max_depth': 20, 'min_samples_split': 12, 'min_samples_leaf': 15, 'max_features': None}
최고 R2 점수 : 0.8053317612878743


### RandomForestRegressor 하이퍼 파라미터 튜닝

In [13]:
def objective(trial) :
    params = {
        # 트리의 개수
        # 너무 적으면 성능이 낮고, 너무 많으면 계산 시간이 오래 걸린다.
        'n_estimators' : trial.suggest_int('n_estimators', 100, 500),
        # 각 트리의 최대 깊이
        'max_depth' : trial.suggest_int('max_depth', 3, 20),
        # 노드 분할을 위한 최소 샘플 수
        'min_samples_split' : trial.suggest_int('min_samples_split', 2, 20),
        # 리프 노드에 필요한 최소 샘플 수
        'min_samples_leaf' : trial.suggest_int('min_samples_leaf', 1, 20),
        # 최적의 분할을 찾기 위해 고려할 피처의 개수
        'max_features' : trial.suggest_categorical('max_features', [None, 'sqrt', 'log2']),
        # 중복을 허용한 샘플링 사용 여부
        'bootstrap' : trial.suggest_categorical('bootstrap', [True, False]),
        # 모든 CPU 사용
        'n_jobs' : -1,
        # 랜덤 시드
        'random_state' : 42
    }

    model = RandomForestRegressor(**params)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    score = cross_val_score(model, X_scaled, y, cv=kf, scoring='r2').mean()
    return score

sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, pruner=MedianPruner(n_warmup_steps=10))
study.optimize(objective, n_trials=50, show_progress_bar=True)

rf_params = study.best_params.copy()
rf_score = study.best_value

print(f'최적 파라미터 : {rf_params}')
print(f'최고 R2 점수 : {rf_score}')

  0%|          | 0/50 [00:00<?, ?it/s]

최적 파라미터 : {'n_estimators': 359, 'max_depth': 20, 'min_samples_split': 7, 'min_samples_leaf': 6, 'max_features': None, 'bootstrap': True}
최고 R2 점수 : 0.8298293806581636


### GradientBoostRegressor 하이퍼 파라미터 튜닝

In [14]:
def objective(trial) :
    params = {
        # 트리의 개수
        # 너무 많으면 과적합, 너무 적으면 학습 부족
        'n_estimators' : trial.suggest_int('n_estimators', 100, 500),
        # 학습률
        # 적으면 정교하지만 많은 트리가 필요함
        'learning_rate' : trial.suggest_float('learning_rate', 0.001, 0.2, log=True),
        # 개별 트리의 최대 깊이
        # 부스팅에서는 깊지 않은 트리를 많이 두는 것이 효과적이다.
        'max_depth' : trial.suggest_int('max_depth', 3, 10),
        # 노드 분할을 위한 최소 샘플 수
        'min_samples_split' : trial.suggest_int('min_samples_split', 2, 20),
        # 리프 노드에 필요한 최소 샘플 수
        'min_samples_leaf' : trial.suggest_int('min_samples_leaf', 1, 20),
        # 각 트리학습에 사용할 데이터 비율
        'subsample' : trial.suggest_float('subsample', 0.5, 1.0),
        # 최적 분할을 위해 사용할 피처의 비율
        'max_features' : trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        # 랜덤 시드
        'random_state' : 42
    }

    model = GradientBoostingRegressor(**params)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    score = cross_val_score(model, X_scaled, y, cv=kf, scoring='r2').mean()
    return score

sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, pruner=MedianPruner(n_warmup_steps=10))
study.optimize(objective, n_trials=50, show_progress_bar=True)

gb_params = study.best_params.copy()
gb_score = study.best_value

print(f'최적 파라미터 : {gb_params}')
print(f'최고 R2 점수 : {gb_score}')

  0%|          | 0/50 [00:00<?, ?it/s]

최적 파라미터 : {'n_estimators': 148, 'learning_rate': 0.06927499365137466, 'max_depth': 7, 'min_samples_split': 16, 'min_samples_leaf': 19, 'subsample': 0.5809488368401539, 'max_features': None}
최고 R2 점수 : 0.8388560177529361


### LGBMRegressor 하이퍼 파라미터 튜닝

In [15]:
def objective(trial) :
    params = {
        # 트리의 개수
        'n_estimators' : trial.suggest_int('n_estimators', 100, 1000),
        # 학습률
        'learning_rate' : trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
        # 트리의 최대 깊이
        'max_depth' : trial.suggest_int('max_depth', 3, 15),
        # 하나의 트리가 가질 수 있는 최대 리프 수 (LGBM의 핵심 파라미터)
        'num_leaves' : trial.suggest_int('num_leaves', 2, 256),
        # 리프 노드가 되기 위한 최소 데이터 수 (과적합 제어)
        'min_child_samples' : trial.suggest_int('min_child_samples', 5, 100),
        # 데이터 샘플링 비율
        'subsample' : trial.suggest_float('subsample', 0.5, 1.0),
        # 개별 트리 학습 시 무작위로 선택되는 피처 비율
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        # L1 규제 강도
        'reg_alpha' : trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        # L2 규제 강도
        'reg_lambda' : trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
        # cpu 코어 최대 활용
        'n_jobs' : -1,
        # 랜덤시트
        'random_state' : 42,
        # 로그 출력 안하게..
        'verbosity' : -1
    }

    model = LGBMRegressor(**params)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    score = cross_val_score(model, X_scaled, y, cv=kf, scoring='r2').mean()
    return score

sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, pruner=MedianPruner(n_warmup_steps=10))
study.optimize(objective, n_trials=50, show_progress_bar=True)

lgbm_params = study.best_params.copy()
lgbm_score = study.best_value

print(f'최적 파라미터 : {lgbm_params}')
print(f'최고 R2 점수 : {lgbm_score}')

  0%|          | 0/50 [00:00<?, ?it/s]

최적 파라미터 : {'n_estimators': 275, 'learning_rate': 0.062482541175872326, 'max_depth': 9, 'num_leaves': 90, 'min_child_samples': 45, 'subsample': 0.9319818188945931, 'colsample_bytree': 0.7778972539293485, 'reg_alpha': 0.029514153215150038, 'reg_lambda': 0.009023643309113781}
최고 R2 점수 : 0.8402474291914865


### XGBoost Regressor 하이퍼 파라미터 튜닝

In [17]:
def objective(trial) :
    params = {
        # GPU 사용 설정
        'device' : 'cuda',
        'tree_method' : 'hist',
        # 트리의 개수
        'n_estimators' : trial.suggest_int('n_estimators', 100, 1000),
        # 학습률
        'learning_rate' : trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
        # 트르의 최대 깊이
        'max_depth' : trial.suggest_int('max_depth', 3, 10),
        # 가중치 합의 최소치 (과적합 제어)
        'min_child_weight' : trial.suggest_int('min_child_weight', 1, 10),
        # 데이터 샘플링 비율
        'subsample' : trial.suggest_float('subsample', 0.5, 1.0),
        # 트리 생성시 사용할 피처 비율
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        # 노드 분할을 위한 최소 손실 감소 값
        'gamma' : trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        # L1 규제 강도
        'reg_alpha' : trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        # L2 규제 강도
        'reg_lambda' : trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        # 랜덤시드
        'random_state' : 42,
        'verbosity' : 0,
    }

    model = XGBRegressor(**params)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    score = cross_val_score(model, X_scaled, y, cv=kf, scoring='r2').mean()
    return score

sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, pruner=MedianPruner(n_warmup_steps=10))
study.optimize(objective, n_trials=50, show_progress_bar=True)

xgb_params = study.best_params.copy()
xgb_score = study.best_value

print(f'최적 파라미터 : {xgb_params}')
print(f'최고 R2 점수 : {xgb_score}')

  0%|          | 0/50 [00:00<?, ?it/s]

최적 파라미터 : {'n_estimators': 790, 'learning_rate': 0.07290966282108481, 'max_depth': 6, 'min_child_weight': 6, 'subsample': 0.6642080088862329, 'colsample_bytree': 0.9012259236600972, 'gamma': 0.030593368645669947, 'reg_alpha': 8.799979362325798, 'reg_lambda': 6.193434549890812}
최고 R2 점수 : 0.838471102327085


### BayesianRidge 하이퍼 파라미터 튜닝

In [20]:
def objective(trial) :
    params = {
        # 최복 반복 횟수
        'max_iter' : trial.suggest_int('max_iter', 100, 1000),
        # 허용 오차
        'tol' : trial.suggest_float('tol', 1e-6, 1e-2, log=True),
        # 감마 분포 파라미터 (정규화 관련)
        'alpha_1' : trial.suggest_float('alpha_1', 1e-8, 1e-2, log=True),
        'alpha_2' : trial.suggest_float('alpha_2', 1e-8, 1e-2, log=True),
        # 가중치 정밀도에 대한 파라미터
        'lambda_1' : trial.suggest_float('lambda_1', 1e-8, 1e-2, log=True),
        'lambda_2' : trial.suggest_float('lambda_2', 1e-8, 1e-2, log=True),
        # 절편 계산 여부
        'fit_intercept' : trial.suggest_categorical('fit_intercept', [True, False])
    }

    model = BayesianRidge(**params)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    score = cross_val_score(model, X_scaled, y, cv=kf, scoring='r2').mean()
    return score

sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, pruner=MedianPruner(n_warmup_steps=10))
study.optimize(objective, n_trials=50, show_progress_bar=True)

br_params = study.best_params.copy()
br_score = study.best_value

print(f'최적 파라미터 : {br_params}')
print(f'최고 R2 점수 : {br_score}')

  0%|          | 0/50 [00:00<?, ?it/s]

최적 파라미터 : {'max_iter': 819, 'tol': 9.534392667816801e-05, 'alpha_1': 1.49310808660669e-06, 'alpha_2': 2.0492023060949682e-07, 'lambda_1': 0.009965544658508204, 'lambda_2': 1.9044737441968943e-06, 'fit_intercept': True}
최고 R2 점수 : 0.8293749850761113


### CatBoost Regressor 하이퍼 파라미터 튜닝

In [24]:
def objective(trial) :
    params = {
        # gpu 설정
        'task_type' : 'GPU',
        'devices' : '0',
        # 트리의 개수
        'iterations' : trial.suggest_int('iterations', 100, 1000),
        # 학습률
        'learning_rate' : trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
        # 트리의 깊이
        'depth' : trial.suggest_int('depth', 4, 10),
        # L2 규제 강도
        'l2_leaf_reg' : trial.suggest_float('l2_leaf_reg', 1e-2, 10.0, log=True),
        # 분할 선택시 무작위성 강도 (과적합 방지)
        'random_strength' : trial.suggest_float('random_strength', 1e-8, 10.0, log=True),
        # 배깅 강도(0이면 배깅 없음, 높을 수록 무작위성 증가)
        'bagging_temperature' : trial.suggest_float('bagging_temperature', 0.0, 1.0),
        # 수치형 피처를 나는 최소 단위 (GPU 사용시 중요)
        'border_count' : trial.suggest_int('border_count', 32, 255),
        # 랜덤 시드
        'random_state' : 42,
        # 로그 출력 안되게..
        'verbose' : False,
        'allow_writing_files' : False
    }

    model = CatBoostRegressor(**params)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    score = cross_val_score(model, X_scaled, y, cv=kf, scoring='r2').mean()
    return score

sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, pruner=MedianPruner(n_warmup_steps=10))
study.optimize(objective, n_trials=50, show_progress_bar=True)

cat_params = study.best_params.copy()
cat_score = study.best_value

print(f'최적 파라미터 : {cat_params}')
print(f'최고 R2 점수 : {cat_score}')

  0%|          | 0/50 [00:00<?, ?it/s]

최적 파라미터 : {'iterations': 914, 'learning_rate': 0.09071323097849458, 'depth': 10, 'l2_leaf_reg': 0.010372443311347384, 'random_strength': 3.5357575714365087e-06, 'bagging_temperature': 0.863088365265509, 'border_count': 64}
최고 R2 점수 : 0.8422217766728034


### 종합 정리

In [26]:
score_list = [
    knn_score, lr_score, ridge_score, lasso_score, en_score, svr_score, dt_score,
    rf_score, gb_score, lgbm_score, xgb_score, br_score, cat_score
]

model_names = [
    'KNeighborsRegressor', 'LinearRegression', 'Ridge', 'Lasso', 'ElasticNet',
    'SVR', 'DecisionTreeRegressor', 'RandomForestRegressor', 'GradientBoostingRegressor',
    'LGBMRegressor', 'XGBRegressor', 'BayesianRidge', 'CatBoostRegressor'
]

score_df = pd.DataFrame(score_list, index=model_names, columns=['score'])
score_df.sort_values('score', ascending=False, inplace=True)
score_df

,score
CatBoostRegressor,0.842222
LGBMRegressor,0.840247
GradientBoostingRegressor,0.838856
XGBRegressor,0.838471
RandomForestRegressor,0.829829
ElasticNet,0.829378
Ridge,0.829376
Lasso,0.829375
BayesianRidge,0.829375
LinearRegression,0.829133


In [27]:
best_model = CatBoostRegressor(**cat_params)
best_model.fit(X_scaled, y)

0:	learn: 1.3229420	total: 11.5ms	remaining: 10.5s
1:	learn: 1.2375740	total: 16.8ms	remaining: 7.65s
2:	learn: 1.1608710	total: 20.7ms	remaining: 6.29s
3:	learn: 1.0933735	total: 24.8ms	remaining: 5.64s
4:	learn: 1.0338533	total: 28.8ms	remaining: 5.24s
5:	learn: 0.9803775	total: 33.4ms	remaining: 5.06s
6:	learn: 0.9341827	total: 37.9ms	remaining: 4.91s
7:	learn: 0.8930450	total: 42.2ms	remaining: 4.78s
8:	learn: 0.8570277	total: 46.3ms	remaining: 4.65s
9:	learn: 0.8248652	total: 52ms	remaining: 4.7s
10:	learn: 0.7965648	total: 58.6ms	remaining: 4.81s
11:	learn: 0.7718618	total: 62.9ms	remaining: 4.73s
12:	learn: 0.7502772	total: 67.2ms	remaining: 4.66s
13:	learn: 0.7311327	total: 71.1ms	remaining: 4.57s
14:	learn: 0.7136709	total: 74.8ms	remaining: 4.49s
15:	learn: 0.6979460	total: 78.8ms	remaining: 4.42s
16:	learn: 0.6838765	total: 82.9ms	remaining: 4.37s
17:	learn: 0.6717249	total: 86.7ms	remaining: 4.31s
18:	learn: 0.6599537	total: 90.3ms	remaining: 4.25s
19:	learn: 0.6498066	tota

In [28]:
with open('model/project2.dat', 'wb') as fp :
    pickle.dump(best_model, fp)
    pickle.dump(scaler1, fp)